In [0]:
%sql
--- Vamos a crea el catalogo
CREATE CATALOG IF NOT EXISTS open_meteo

In [0]:
%sql
--- Vamos a crear el shema
CREATE SCHEMA IF NOT EXISTS open_meteo.api_open

In [0]:
import requests
import json
from datetime import datetime, timezone

#Cordenadas  aproximadas de Sanjose Costa Rica
latitude = 9.9281
longitude = -84.0907

url = "https://api.open-meteo.com/v1/forecast"

params = {
  "latitude" :  latitude,
  "longitude" : longitude,
  "hourly" : (
      "temperature_2m,"
      "relative_humidity_2m,"
      "precipitation,"
      "wind_speed_10m"
  ),
   "forecast_days": 7,
   "timezone": "America/Costa_Rica"

}

response = requests.get(url, params=params, timeout=30)

# Verificar que la solicitud fue exitosa
response.raise_for_status()

weather_data = response.json()

print("Extracción exitosa")
print("Ciudad: San José, Costa Rica")
print("Zona horaria:", weather_data["timezone"])
print("Cantidad de horas:",
      len(weather_data["hourly"]["time"]))


In [0]:
from pyspark.sql import functions as F

# Guardar la respuesta JSON original
raw_record = [{
    "city": "San Jose",
    "source": "Open-Meteo",
    "extracted_at": datetime.now(timezone.utc).isoformat(),
    "payload": json.dumps(weather_data)
}]

catalog = "open_meteo"
shema = "api_open"

bronze_df = spark.createDataFrame(raw_record)

bronze_table = f"{catalog}.{shema}.bronze_weather_raw"

bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(bronze_table)

print("Datos guardados en Bronze")

In [0]:
%sql
select *
from open_meteo.api_open.bronze_weather_raw